### 1. Preprocesamiento y Limpieza de Datos

Para cumplir con las pautas metodológicas de la tarea, se realizó una inspección y limpieza del dataset de partidos del fútbol uruguayo:
*   **Eliminación de columnas constantes:** Se descartaron las variables `competition`, `level`, `continent`, `home_country`, `away_country`, `home_code`, `away_code`, `home_continent` y `away_continent` debido a que presentan varianza cero (un único valor constante para todo el registro histórico), aportando valor predictivo nulo.
*   **Duplicados:** Se eliminaron las filas duplicadas exactas para evitar redundancias e inconsistencias en el entrenamiento.
*   **Tratamiento del Target:** Se construyó la variable objetivo `ganador` ('L' si gana el local, 'V' si gana el visitante, 'E' en caso de empate) comparando los goles anotados (`gh` y `ga`). Los partidos definidos por penales ('P') se trataron estadísticamente como empates ('E') para evitar que la alta varianza de los penales distorsione el rendimiento de los modelos en el tiempo reglamentario.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# 1. Carga de datos
file_path = '../data/raw/futbol_uruguayo.csv'
df = pd.read_csv(file_path)

# 2. Limpieza básica y creación del Target (y)
df = df.drop_duplicates()
df['date'] = pd.to_datetime(df['date'])

def asignar_ganador(row):
    if row['gh'] > row['ga']: return 'L'
    elif row['gh'] < row['ga']: return 'V'
    else: return 'E'

df['ganador'] = df.apply(asignar_ganador, axis=1)

# ORDENAMIENTO CRONOLÓGICO (Fundamental para datos temporales)
df = df.sort_values('date').reset_index(drop=True)

# 3. Separación de X e y eliminando las columnas que revelan el resultado
y = df['ganador']
X = df.drop(columns=['ganador', 'gh', 'ga', 'full_time'])

# 4. División Train/Test Temporal (como se solicitó)
mask_train = X['date'].dt.year <= 2023
mask_test = X['date'].dt.year >= 2024

X_train, y_train = X[mask_train].copy(), y[mask_train].copy()
X_test, y_test = X[mask_test].copy(), y[mask_test].copy()

### 2. División Temporal y Prevención de Data Leakage

Al trabajar con series temporales (partidos de fútbol desde 1932 hasta 2025), la partición de los datos debe respetar estrictamente la línea cronológica para evitar la fuga de información (*data leakage*):
*   La columna `date` se convirtió al formato datetime y el dataset se ordenó de forma cronológica estricta.
*   **Conjunto de Entrenamiento:** Partidos jugados hasta el año 2023 inclusive.
*   **Conjunto de Evaluación (Test):** Partidos jugados exclusivamente durante los años 2024 y 2025.

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Transformador personalizado para eliminar columnas
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop, errors='ignore')

# Transformador para manejar fechas (extrae año y mes, y elimina la fecha original)
class DateTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_new = X.copy()
        X_new['year'] = X_new['date'].dt.year
        X_new['month'] = X_new['date'].dt.month
        return X_new.drop(columns=['date'])

# Construimos el Pipeline
columnas_a_eliminar = ['competition', 'level', 'continent', 'home_country',
                       'away_country', 'home_code', 'away_code',
                       'home_continent', 'away_continent', 'home_ident', 'away_ident']

preprocessing_pipeline = Pipeline([
    ('dropper', DropColumns(columnas_a_eliminar)),
    ('date_features', DateTransformer()),
    # Aplicamos OneHotEncoder a los equipos (home y away) para transformarlos en números
    ('encoder', ColumnTransformer([
        ('teams_encoder', OneHotEncoder(handle_unknown='ignore'), ['home', 'away'])
    ], remainder='passthrough'))
])

# ¡Ahora puedes preparar tus datos en una sola línea!
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

# --- VISUALIZAR LA SALIDA DEL PIPELINE ---

# 1. Recuperamos los nuevos nombres de las columnas generados por el OneHotEncoder
nomi_colonne = preprocessing_pipeline.named_steps['encoder'].get_feature_names_out()

# 2. Transformamos la matriz resultante en un DataFrame Pandas legible
# (usamos .toarray() porque el encoder produce una "matriz dispersa" comprimida)
X_train_visibile = pd.DataFrame(
    X_train_processed.toarray(),
    columns=nomi_colonne
)

print("📌 ¡AQUÍ ESTÁN LOS DATOS DESPUÉS DEL PIPELINE (LISTOS PARA RANDOM FOREST Y NAIVE BAYES)!")
print(f"Número de columnas creadas: {X_train_visibile.shape[1]}")
display(X_train_visibile.head())

📌 ¡AQUÍ ESTÁN LOS DATOS DESPUÉS DEL PIPELINE (LISTOS PARA RANDOM FOREST Y NAIVE BAYES)!
Número de columnas creadas: 78


,teams_encoder__home_Albion,teams_encoder__home_Bella Vista,teams_encoder__home_Boston River,teams_encoder__home_CA Basanez,teams_encoder__home_CA Cerro,teams_encoder__home_CA Fenix,teams_encoder__home_CA Juventud,teams_encoder__home_CA Penarol,teams_encoder__home_CA Progreso,teams_encoder__home_CSyd Villa Espanola,...,teams_encoder__away_Racing Club,teams_encoder__away_Rampla Juniors Futbol Club,teams_encoder__away_Rentistas,teams_encoder__away_River Plate,teams_encoder__away_Rocha Futbol Club,teams_encoder__away_Tacuarembo Futbol Club,teams_encoder__away_Torque FC,teams_encoder__away_Villa Teresa,remainder__year,remainder__month
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1932.0,3.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1932.0,3.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1932.0,3.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1932.0,3.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1932.0,3.0


### 3. Pipeline de Preprocesamiento y Clasificador Base

*   **Pipeline Reproducible:** Se diseñó un flujo estructurado utilizando transformadores personalizados y la clase `ColumnTransformer` de `scikit-learn`. Esto permite extraer características de las fechas (año y mes) y aplicar codificación One-Hot (`OneHotEncoder`) a los nombres de los equipos de forma limpia y automatizada.
*   **Clasificador Base (Baseline):** Se implementó un modelo de referencia personalizado que predice la victoria basándose en la mayor proporción de partidos ganados por cada equipo en los **últimos 10 años** anteriores a la fecha del encuentro analizado, cumpliendo con los requerimientos de la línea base y sirviendo de punto de comparación para los algoritmos avanzados (Árboles de Decisión y Naive Bayes).

In [3]:
from sklearn.base import BaseEstimator, ClassifierMixin

class ClasificadorBase10Anios(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        # Unimos X e y para tener el historial completo
        self.historia_ = X.copy()
        self.historia_['target'] = y.copy()
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        predicciones = []

        # Iterar es inevitable para la ventana temporal deslizante, pero usamos consultas optimizadas
        for idx, row in X.iterrows():
            fecha_partido = row['date']
            fecha_limite = fecha_partido - pd.DateOffset(years=10)
            eq_local = row['home']
            eq_visit = row['away']

            # Filtro rápido sobre el DataFrame histórico
            mask_tiempo = (self.historia_['date'] >= fecha_limite) & (self.historia_['date'] < fecha_partido)
            historial = self.historia_[mask_tiempo]

            # Estadísticas del equipo local
            partidos_loc = historial[(historial['home'] == eq_local) | (historial['away'] == eq_local)]
            vict_loc = len(partidos_loc[(partidos_loc['home'] == eq_local) & (partidos_loc['target'] == 'L')]) + \
                       len(partidos_loc[(partidos_loc['away'] == eq_local) & (partidos_loc['target'] == 'V')])
            prop_loc = vict_loc / len(partidos_loc) if len(partidos_loc) > 0 else 0

            # Estadísticas del equipo visitante
            partidos_vis = historial[(historial['home'] == eq_visit) | (historial['away'] == eq_visit)]
            vict_vis = len(partidos_vis[(partidos_vis['home'] == eq_visit) & (partidos_vis['target'] == 'L')]) + \
                       len(partidos_vis[(partidos_vis['away'] == eq_visit) & (partidos_vis['target'] == 'V')])
            prop_vis = vict_vis / len(partidos_vis) if len(partidos_vis) > 0 else 0

            # Predicción
            if prop_loc > prop_vis:
                predicciones.append('L')
            elif prop_vis > prop_loc:
                predicciones.append('V')
            else:
                predicciones.append('E')

        return np.array(predicciones)

# --- PROBAR Y VISUALIZAR EL CLASIFICADOR BASE ---

print("\n⚙️ Entrenando el Clasificador Base...")
# 1. Creamos el modelo y le damos los datos históricos (X_train SIN procesar por el pipeline)
clf_base = ClasificadorBase10Anios()
clf_base.fit(X_train, y_train)

# 2. Tomamos solo los primeros 5 partidos de 2024 para una prueba visual rápida
X_test_campione = X_test.head(5)
y_test_campione = y_test.head(5)

# 3. Hacemos que el modelo realice las predicciones
previsiones_base = clf_base.predict(X_test_campione)

# 4. Creamos una bonita tabla resumen para comparar los datos
tabella_risultati = X_test_campione[['date', 'home', 'away']].copy()
tabella_risultati['Victorias_Predichas (Modelo)'] = previsiones_base
tabella_risultati['Victorias_Reales (Target)'] = y_test_campione.values

print("📌 PREDICCIONES DEL CLASIFICADOR BASE SOBRE LOS PRIMEROS 5 PARTIDOS DEL CONJUNTO DE PRUEBA:")
display(tabella_risultati)


⚙️ Entrenando el Clasificador Base...
📌 PREDICCIONES DEL CLASIFICADOR BASE SOBRE LOS PRIMEROS 5 PARTIDOS DEL CONJUNTO DE PRUEBA:


,date,home,away,Victorias_Predichas (Modelo),Victorias_Reales (Target)
14734,2024-02-17,Nacional,River Plate,L,L
14735,2024-02-17,CA Fenix,Danubio,V,V
14736,2024-02-17,Miramar Misiones,CA Progreso,V,V
14737,2024-02-18,Deportivo Maldonado,Boston River,E,V
14738,2024-02-18,CA Cerro,Montevideo Wanderers,V,E
